# 02 — Feature Engineering

**Purpose:** Build creator-level and post-level features that a partnerships team would use to evaluate, compare, and shortlist creators.

Features built in this notebook:
- Sponsored vs organic engagement rates
- Sponsored engagement lift/drop
- Posting consistency score
- Brand mention diversity
- Post-level time and content features

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import load_creators, load_posts
from src.clean import clean_creators, clean_posts
from src.features import add_post_features, build_creator_features
from src.utils import set_plot_style, COLORS, TIER_ORDER

set_plot_style()

In [ ]:
creators = clean_creators(load_creators())
posts = clean_posts(load_posts())

# Add post-level features
posts = add_post_features(posts)
print(f"Post features added: {list(posts.columns[-6:])}")

# Build creator-level features
creators_enriched = build_creator_features(posts, creators)
print(f"Creator features added: {[c for c in creators_enriched.columns if c not in creators.columns]}")

## 1. Sponsored vs Organic Engagement by Creator

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Only creators with both sponsored and organic posts
has_both = creators_enriched[
    (creators_enriched['sponsored_count'] >= 3) &
    (creators_enriched['organic_count'] >= 3)
].copy()

print(f"Creators with 3+ sponsored AND 3+ organic posts: {len(has_both)}")

ax.scatter(
    has_both['organic_er'], has_both['sponsored_er'],
    alpha=0.5, color=COLORS['primary'], s=30
)
lim = max(has_both['organic_er'].max(), has_both['sponsored_er'].max()) + 1
ax.plot([0, lim], [0, lim], '--', color=COLORS['neutral'], alpha=0.5, label='Parity line')
ax.set_xlabel('Organic Engagement Rate (%)')
ax.set_ylabel('Sponsored Engagement Rate (%)')
ax.set_title('Sponsored vs Organic Engagement Rate by Creator')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Sponsored Lift Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
lift = has_both['sponsored_lift'].clip(-100, 100)
ax.hist(lift, bins=40, color=COLORS['primary'], alpha=0.7, edgecolor='white')
ax.axvline(0, color=COLORS['danger'], linestyle='--', label='No change')
ax.axvline(lift.median(), color=COLORS['accent'], linestyle='-', label=f'Median: {lift.median():.1f}%')
ax.set_xlabel('Sponsored Engagement Lift (%)')
ax.set_ylabel('Creators')
ax.set_title('Distribution of Sponsored Content Engagement Lift')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Creators with positive sponsored lift: {(has_both['sponsored_lift'] > 0).sum()} ({(has_both['sponsored_lift'] > 0).mean()*100:.1f}%)")
print(f"Creators with negative sponsored lift: {(has_both['sponsored_lift'] < 0).sum()} ({(has_both['sponsored_lift'] < 0).mean()*100:.1f}%)")

## 3. Posting Consistency Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Consistency score distribution
axes[0].hist(creators_enriched['consistency_score'], bins=30, color=COLORS['success'], alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Consistency Score (0-100)')
axes[0].set_ylabel('Creators')
axes[0].set_title('Posting Consistency Score Distribution')

# Consistency vs engagement
axes[1].scatter(
    creators_enriched['consistency_score'],
    creators_enriched['avg_engagement_rate'],
    alpha=0.4, s=20, color=COLORS['secondary']
)
axes[1].set_xlabel('Consistency Score')
axes[1].set_ylabel('Avg Engagement Rate (%)')
axes[1].set_title('Consistency vs Engagement Rate')

plt.tight_layout()
plt.show()

## 4. Engagement Rate Buckets

In [ ]:
er_dist = posts.groupby(['er_bucket', 'is_sponsored']).size().unstack(fill_value=0)
er_dist.columns = ['Organic', 'Sponsored']

fig, ax = plt.subplots(figsize=(10, 4))
er_dist.plot(kind='bar', ax=ax, color=[COLORS['primary'], COLORS['accent']])
ax.set_title('Post Count by Engagement Rate Bucket')
ax.set_xlabel('Engagement Rate Bucket')
ax.set_ylabel('Posts')
ax.legend(title='Post Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Feature Summary Table

Enriched creator dataset — ready for scoring in Notebook 04.

In [ ]:
feature_cols = [
    'creator_id', 'category', 'follower_tier', 'followers',
    'avg_engagement_rate', 'sponsored_er', 'organic_er', 'sponsored_lift',
    'sponsored_count', 'organic_count', 'consistency_score', 'brand_mention_count'
]
creators_enriched[feature_cols].describe().round(2)

## Summary

**Key findings from feature engineering:**
- Sponsored content shows a median engagement drop, but a meaningful subset of creators maintain or exceed organic performance
- Posting consistency varies widely — this is a useful differentiator for shortlisting
- The enriched creator dataset now has 7 new features ready for scoring

**Next step:** Sponsored vs organic deep-dive (Notebook 03)